# QAMP Q1: anchor facts of tokenised-asset contracts (Ethereum mainnet)

Reads, for the tokens fixed in `qamp/data/sample_tokens.txt` at the plan tag below, **observations only**:
verified source (token and implementation), EIP-1967 admin and implementation slots, `owner()`, `defaultAdmin()`,
role members of AccessControlEnumerable roles, and for every controlling address: contract or EOA, nonce, Safe
threshold and owners (one hop), timelock delay. Nothing is classified here: `python -m qamp facts` does that from the
committed file (`src/qamp/extraction.py`, `facts_from_raw`).

**How to use:** set `REPO_REF` to the tag that fixes the sample (create the tag *before* running). Then *Runtime, Run
all*. Paste a **new** Etherscan API key when asked (hidden; never printed or saved). At the end `qamp_q1_data.zip`
downloads. Unzip it into `qamp/data/`, commit, then delete the key on etherscan.io/apidashboard.


In [ ]:
# Cell 1: setup (edit REPO_REF only)
REPO_URL = "https://github.com/FahdSaif/RWA-Asset-tokenization.git"
REPO_REF = "qamp-q1-plan"          # tag that fixes qamp/data/sample_tokens.txt and the extraction code

import subprocess, sys, os
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, "repo"], check=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "pycryptodome"], check=True)
sys.path.insert(0, "repo/qamp/src")
commit = subprocess.run(["git", "-C", "repo", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("plan tag", REPO_REF, "commit", commit[:10])


In [ ]:
# Cell 2: run (nothing to edit)
import datetime as dt, getpass, json, zipfile
from pathlib import Path
import requests
from Crypto.Hash import keccak
from qamp.extraction import Observer, digest, selector

ROLE_NAMES = ["DEFAULT_ADMIN_ROLE", "MINTER_ROLE", "BURNER_ROLE", "PAUSER_ROLE", "UNPAUSER_ROLE", "FREEZER_ROLE",
              "UPGRADER_ROLE", "CONFIGURER_ROLE", "LIST_CONFIGURER_ROLE", "SUPPLY_MANAGER_ROLE", "EMERGENCY_ROLE",
              "ROLE_ISSUER", "ROLE_TRANSFER_AGENT", "ROLE_MASTER", "AGENT_ROLE", "OPERATOR_ROLE", "COMPLIANCE_ROLE"]
k256 = lambda b: keccak.new(digest_bits=256, data=b).digest()
ROLE_SEL = {n: selector(n + "()", k256) for n in ROLE_NAMES}

def read_sample(p):
    out = []
    for line in Path(p).read_text().splitlines():
        if line.strip() and not line.startswith("#"):
            out.append(line.split()[0].lower())
    return out

TOKENS = read_sample("repo/qamp/data/sample_tokens.txt")
print(len(TOKENS), "tokens in the fixed sample")

key = getpass.getpass("Paste your Etherscan API key (hidden): ").strip()
def get_json(u):
    try:
        r = requests.get(u, timeout=60); r.raise_for_status(); return r.json()
    except requests.RequestException as e:
        raise RuntimeError(f"network error: {type(e).__name__}") from None   # never show the URL: it holds the key

ob = Observer(get_json, key)
block = ob.block()
retrieved = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
out = Path("qamp_q1"); (out / "sources").mkdir(parents=True, exist_ok=True)
records = []
for i, a in enumerate(TOKENS, 1):
    rec = ob.observe_token(a, ROLE_SEL)
    for k in ("source", "implementation_source"):
        if rec.get(k):
            (out / "sources" / f"{a}{'.impl' if k.startswith('impl') else ''}.sol.txt").write_text(rec[k])
            rec[k] = f"sources/{a}{'.impl' if k.startswith('impl') else ''}.sol.txt"
    records.append(rec)
    print(f"{i:3d} {a[:10]}... {str(rec['contract_name'])[:22]:22s} proxy={rec['is_proxy']!s:5} controllers={len(rec['controllers'])} roles={len(rec['roles'])} errors={len(rec['errors'])}")
del key
raw = dict(tokens=records, block=block, retrieved_at_utc=retrieved, plan_ref=REPO_REF, plan_commit=commit, calls=ob.calls)
(out / "q1_raw.json").write_text(json.dumps(raw, indent=1))
summary = dict(n_tokens=len(records), with_errors=sum(bool(r["errors"]) for r in records), block=block,
               retrieved_at_utc=retrieved, plan_ref=REPO_REF, plan_commit=commit, sha256_raw=digest(raw))
(out / "q1_summary.json").write_text(json.dumps(summary, indent=1))
print(json.dumps(summary, indent=1))
with zipfile.ZipFile("qamp_q1_data.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in out.rglob("*"):
        z.write(p, p.relative_to(out))
try:
    from google.colab import files; files.download("qamp_q1_data.zip")
except ImportError:
    print("wrote qamp_q1_data.zip")
